In [ ]:
"""resnet_cbam_classification.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1F7CYECUPFsn3WsXI58EuAirlGwDhC63j

resnet_cbam_classification.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1cKqvEurvmbI_XqLTKf4ygOLl93ofs4JL

ResNet-CBAM Advanced CNN Classification - Version Multi-Méthodes
Classification haute performance pour TOUTES les méthodes (GASF, MTF, RP)
"""

In [ ]:
!pip install tensorflow keras scipy scikit-learn imbalanced-learn  google.colab
!pip install requests pandas matplotlib seaborn tqdm numpy pywavelets neurokit2
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import seaborn as sns
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.flush_and_unmount()  # Unmount if mounted
!rm -rf /content/drive  # Clear any remaining files
drive.mount('/content/drive')

In [ ]:
# Configuration paths (à adapter selon votre structure)
BASE_PATH = '/content/drive/MyDrive/SeizeIT2_Project/'
IMAGE_DATA_PATH = BASE_PATH + 'image_data/'
GAN_OUTPUT_PATH = BASE_PATH + 'gan_generated/'
PLOTS_PATH = BASE_PATH + 'plots/'
DATA_SPLIT_PATH = BASE_PATH + 'data_split/'
MODELS_PATH = BASE_PATH + 'models/'

In [ ]:
for path in [MODELS_PATH, PLOTS_PATH]:
    os.makedirs(path, exist_ok=True)

In [ ]:
print("🔧 Configuration terminée!")

In [ ]:
def cbam_block(input_feature, ratio=8, name_prefix="cbam"):
    """
    CBAM CORRIGÉ - Utilise uniquement des couches Keras
    """
    channel_axis = -1
    channel = input_feature.shape[channel_axis]

    # CHANNEL ATTENTION avec couches Keras uniquement
    avg_pool = layers.GlobalAveragePooling2D(name=f"{name_prefix}_avg_pool")(input_feature)
    max_pool = layers.GlobalMaxPooling2D(name=f"{name_prefix}_max_pool")(input_feature)

    # MLP partagé
    def shared_mlp(x, suffix):
        x = layers.Dense(channel//ratio, activation='relu', name=f"{name_prefix}_fc1_{suffix}")(x)
        x = layers.Dense(channel//4, activation='relu', name=f"{name_prefix}_fc2_{suffix}")(x)
        x = layers.Dense(channel, name=f"{name_prefix}_fc3_{suffix}")(x)
        return x

    avg_out = shared_mlp(avg_pool, "avg")
    max_out = shared_mlp(max_pool, "max")

    channel_attention = layers.Add(name=f"{name_prefix}_add")([avg_out, max_out])
    channel_attention = layers.Activation('sigmoid', name=f"{name_prefix}_sigmoid")(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    feature = layers.Multiply(name=f"{name_prefix}_channel_mul")([input_feature, channel_attention])

    # SPATIAL ATTENTION CORRIGÉ - Utiliser Lambda layers au lieu de tf.reduce_*
    def spatial_avg_pool(x):
        return tf.reduce_mean(x, axis=3, keepdims=True)

    def spatial_max_pool(x):
        return tf.reduce_max(x, axis=3, keepdims=True)

    avg_pool_spatial = layers.Lambda(spatial_avg_pool, name=f"{name_prefix}_spatial_avg")(feature)
    max_pool_spatial = layers.Lambda(spatial_max_pool, name=f"{name_prefix}_spatial_max")(feature)

    spatial_concat = layers.Concatenate(axis=3, name=f"{name_prefix}_spatial_concat")(
        [avg_pool_spatial, max_pool_spatial]
    )

    # Convolution spatiale simple
    spatial_attention = layers.Conv2D(1, 7, padding='same', activation='sigmoid',
                                     name=f"{name_prefix}_spatial_conv")(spatial_concat)

    output = layers.Multiply(name=f"{name_prefix}_spatial_mul")([feature, spatial_attention])

    return output

In [ ]:
def residual_block(x, filters, stride=1, conv_shortcut=True, name_prefix="res_block"):
    """
    Bloc résiduel AMÉLIORÉ - REMPLACE votre ancien residual_block
    """
    shortcut = x

    # Architecture bottleneck améliorée
    # 1x1 compression
    x = layers.Conv2D(filters//4, 1, use_bias=False, name=f"{name_prefix}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn1")(x)
    x = layers.Activation('relu', name=f"{name_prefix}_relu1")(x)

    # 3x3 convolution principale
    x = layers.Conv2D(filters//4, 3, strides=stride, padding='same', use_bias=False,
                     name=f"{name_prefix}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn2")(x)
    x = layers.Activation('relu', name=f"{name_prefix}_relu2")(x)

    # 1x1 expansion
    x = layers.Conv2D(filters, 1, use_bias=False, name=f"{name_prefix}_conv3")(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn3")(x)

    # CBAM AVANT l'addition (IMPORTANT)
    x = cbam_block(x, name_prefix=f"{name_prefix}_cbam")

    # Shortcut connection améliorée
    if conv_shortcut:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False,
                               name=f"{name_prefix}_shortcut_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name_prefix}_shortcut_bn")(shortcut)

    x = layers.Add(name=f"{name_prefix}_add")([shortcut, x])
    x = layers.Activation('relu', name=f"{name_prefix}_output")(x)

    return x

In [ ]:
def build_simple_cnn(input_shape=(64, 64, 3), num_classes=2):
    """
    ResNet-CBAM AMÉLIORÉ - REMPLACE le CNN simple
    """
    inputs = Input(shape=input_shape)

    # STEM plus robuste pour 64x64
    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.Activation('relu', name='stem_relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same', name='stem_pool')(x)  # 16x16

    # Stage 1: 16x16 -> 16x16 (Blocs résiduel avec CBAM)
    x = residual_block(x, 128, stride=1, name_prefix='stage1_block1')
    x = residual_block(x, 128, stride=1, name_prefix='stage1_block2')

    # Stage 2: 16x16 -> 8x8
    x = residual_block(x, 256, stride=2, name_prefix='stage2_block1')
    x = residual_block(x, 256, stride=1, name_prefix='stage2_block2')
    x = residual_block(x, 256, stride=1, name_prefix='stage2_block3')

    # Stage 3: 8x8 -> 4x4
    x = residual_block(x, 512, stride=2, name_prefix='stage3_block1')
    x = residual_block(x, 512, stride=1, name_prefix='stage3_block2')

    # CBAM global final pour attention sur les features complètes
    x = cbam_block(x, ratio=16, name_prefix='global_cbam')

    # Classification head amélioré avec connexions résiduelles
    x = layers.GlobalAveragePooling2D(name='global_pool')(x)
    x = layers.Dropout(0.5, name='dropout1')(x)

    # Dense layers avec skip connection
    dense1 = layers.Dense(512, activation='relu', name='dense1')(x)
    dense1_drop = layers.Dropout(0.3, name='dropout2')(dense1)

    dense2 = layers.Dense(256, activation='relu', name='dense2')(dense1_drop)
    dense2_drop = layers.Dropout(0.2, name='dropout3')(dense2)

    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(dense2_drop)

    return Model(inputs, outputs, name='ResNetCBAM_Improved')

In [ ]:
def specialized_preprocessing(X_train, X_test, method='gasf'):
    """
    Preprocessing spécialisé CORRIGÉ - évite les erreurs TensorFlow
    """
    print(f"Preprocessing spécialisé pour {method.upper()}...")

    X_train = X_train.astype('float32')
    X_test = X_test.astype('float32')

    if method == 'gasf':
        # GASF: améliorer le contraste des patterns angulaires (VERSION CORRIGÉE)
        print("   Amélioration contraste GASF...")

        # Utiliser NumPy au lieu de TensorFlow pour éviter l'erreur
        def adjust_contrast_numpy(image, contrast_factor=1.3):
            """Fonction de contraste compatible NumPy"""
            # Convertir en float et normaliser
            img = image.astype(np.float32)

            # Calculer la moyenne pour chaque canal
            mean_vals = np.mean(img, axis=(0, 1), keepdims=True)

            # Appliquer l'ajustement de contraste
            adjusted = (img - mean_vals) * contrast_factor + mean_vals

            # Clipper entre 0 et 1
            return np.clip(adjusted, 0, 1)

        # Appliquer à toutes les images
        for i in range(X_train.shape[0]):
            X_train[i] = adjust_contrast_numpy(X_train[i], 1.3)
        for i in range(X_test.shape[0]):
            X_test[i] = adjust_contrast_numpy(X_test[i], 1.3)

    elif method == 'mtf':
        # MTF: normalisation robuste par percentiles (DÉJÀ CORRECT)
        print("   Normalisation robuste MTF...")
        for i in range(X_train.shape[0]):
            img = X_train[i]
            p1, p99 = np.percentile(img, [1, 99])
            X_train[i] = np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)
        for i in range(X_test.shape[0]):
            img = X_test[i]
            p1, p99 = np.percentile(img, [1, 99])
            X_test[i] = np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

    elif method == 'rp':
        # RP: améliorer les patterns de récurrence (DÉJÀ CORRECT)
        print("   Amélioration patterns RP...")
        # Déjà bien normalisé, juste clipper
        X_train = np.clip(X_train, 0, 1)
        X_test = np.clip(X_test, 0, 1)

    print(f"   Preprocessing {method.upper()} terminé")
    return X_train, X_test

In [ ]:
def load_data_for_method(method):
    print(f"Chargement des données pour {method.upper()}...")

    try:
        # Charger TOUTES les données (originales + augmentées)
        X_train_orig = np.load(IMAGE_DATA_PATH + f'{method}/X_train_{method}.npy')
        X_test = np.load(IMAGE_DATA_PATH + f'{method}/X_test_{method}.npy')
        y_train_orig = np.load(DATA_SPLIT_PATH + 'y_train.npy')
        y_test = np.load(DATA_SPLIT_PATH + 'y_test.npy')

        # ✅ CORRECTION 1: Utiliser PLUS de données d'entraînement
        # Prendre une partie des données de test pour l'entraînement
        from sklearn.model_selection import train_test_split

        X_test_for_train, X_test_final, y_test_for_train, y_test_final = train_test_split(
            X_test, y_test, test_size=0.3, random_state=42, stratify=y_test
        )

        # Combiner pour avoir plus de données d'entraînement
        X_train_big = np.concatenate([X_train_orig, X_test_for_train], axis=0)
        y_train_big = np.concatenate([y_train_orig, y_test_for_train], axis=0)

        print(f"   Nouvelles données - Train: {X_train_big.shape}")
        print(f"   Nouvelles données - Test: {X_test_final.shape}")
        print(f"   Distribution train: {np.bincount(y_train_big)}")

        return X_train_big, y_train_big, X_test_final, y_test_final, False

    except Exception as e:
        print(f"Erreur : {e}")
        return None, None, None, None, False

In [ ]:
def preprocess_data(X_train, X_test, augmented=False):
    """
    Préparation des données avec normalisation adaptative
    """
    print("🔧 Préparation des données...")

    # Conversion en float32
    X_train = X_train.astype('float32')
    X_test = X_test.astype('float32')

    print(f"   Avant normalisation:")
    print(f"   Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
    print(f"   Test range: [{X_test.min():.3f}, {X_test.max():.3f}]")

    if augmented:
        # Si données GAN (dans [-1, 1]), les convertir vers [0, 1]
        if X_train.min() < 0:
            X_train = (X_train + 1) / 2.0
            print("   🔄 Conversion GAN [-1,1] → [0,1]")

        if X_test.min() < 0:
            X_test = (X_test + 1) / 2.0

    # S'assurer que tout est dans [0, 1]
    X_train = np.clip(X_train, 0, 1)
    X_test = np.clip(X_test, 0, 1)

    print(f"   Après normalisation:")
    print(f"   Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
    print(f"   Test range: [{X_test.min():.3f}, {X_test.max():.3f}]")

    return X_train, X_test

In [ ]:
def train_and_evaluate_method(method):
    """
    Version CORRIGÉE de la fonction d'entraînement
    """
    print(f"\n{'='*70}")
    print(f"ENTRAÎNEMENT RESNET-CBAM POUR {method.upper()}")
    print(f"{'='*70}")

    # 1. Charger les données
    X_train, y_train, X_test, y_test, is_augmented = load_data_for_method(method)

    if X_train is None:
        print(f"Impossible de charger les données pour {method}")
        return None

    # 2. CORRECTION: Utiliser la version corrigée du preprocessing
    try:
        X_train, X_test = specialized_preprocessing(X_train, X_test, method)
    except Exception as e:
        print(f"Erreur preprocessing spécialisé: {e}")
        print("Utilisation du preprocessing simple...")
        X_train, X_test = simple_preprocessing(X_train, X_test, method)

    # 3. Normalisation finale de sécurité
    if X_train.min() < 0 or X_train.max() > 1:
        X_train = (X_train - X_train.min()) / (X_train.max() - X_train.min())
    if X_test.min() < 0 or X_test.max() > 1:
        X_test = (X_test - X_test.min()) / (X_test.max() - X_test.min())

    print(f"Après normalisation:")
    print(f"   Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
    print(f"   Test range: [{X_test.min():.3f}, {X_test.max():.3f}]")

    # 4. Split validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
    )

    print(f"Splits finaux:")
    print(f"   Train: {X_train_split.shape}")
    print(f"   Validation: {X_val_split.shape}")
    print(f"   Test: {X_test.shape}")

    # 5. Construire le modèle ResNet-CBAM
    print(f"Construction du modèle ResNet-CBAM pour {method.upper()}...")
    model = build_simple_cnn(
        input_shape=X_train.shape[1:],
        num_classes=2
    )

    # 6. Compilation avec paramètres optimisés
    model.compile(
        optimizer=Adam(learning_rate=0.0005, amsgrad=True),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"Modèle compilé avec LR=0.0005 + AMSGrad")
    print(f"Paramètres du modèle: {model.count_params():,}")

    # 7. Callbacks améliorés
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=20,
            restore_best_weights=True,
            verbose=1,
            mode='max'
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.7,
            patience=8,
            min_lr=1e-7,
            verbose=1,
            mode='min',
            cooldown=3
        ),
        ModelCheckpoint(
            filepath=MODELS_PATH + f'resnet_cbam_{method}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1,
            mode='max'
        )
    ]

    # 8. Entraînement
    print(f"Début de l'entraînement ResNet-CBAM...")

    history = model.fit(
        X_train_split, y_train_split,
        validation_data=(X_val_split, y_val_split),
        epochs=80,
        batch_size=24,
        callbacks=callbacks,
        verbose=1,
        shuffle=True
    )

    # 9. Évaluation finale
    print(f"Évaluation finale pour {method.upper()}...")

    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Normal', 'Crise'], output_dict=True)
    cm = confusion_matrix(y_test, y_pred)

    # 10. Résultats
    results = {
        'method': method,
        'accuracy': accuracy,
        'precision_normal': report['Normal']['precision'],
        'recall_normal': report['Normal']['recall'],
        'f1_normal': report['Normal']['f1-score'],
        'precision_crise': report['Crise']['precision'],
        'recall_crise': report['Crise']['recall'],
        'f1_crise': report['Crise']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'is_augmented': is_augmented,
        'history': history.history,
        'confusion_matrix': cm,
        'classification_report': report
    }

    # 11. Affichage
    print(f"\n🎯 RÉSULTATS FINAUX ResNet-CBAM pour {method.upper()}:")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1-macro: {results['f1_macro']:.4f}")
    print(f"   F1-Crise: {results['f1_crise']:.4f}")
    print(f"   Precision-Crise: {results['precision_crise']:.4f}")
    print(f"   Recall-Crise: {results['recall_crise']:.4f}")

    print(f"\nMatrice de confusion:")
    print(cm)

    # 12. Diagnostic
    if accuracy > 0.96:
        print(f"🎉 EXCELLENTE PERFORMANCE! Objectif dépassé!")
    elif accuracy > 0.93:
        print(f"✅ TRÈS BONNE PERFORMANCE!")
    elif accuracy > 0.90:
        print(f"✅ BONNE PERFORMANCE")
    else:
        print(f"⚠️ Performance à améliorer")

    # 13. Sauvegarde
    model.save(MODELS_PATH + f'resnet_cbam_{method}_final.h5')
    print(f"💾 Modèle sauvegardé: resnet_cbam_{method}_final.h5")

    return results

In [ ]:
def compare_all_methods():
    """
    Compare toutes les méthodes et génère un rapport complet
    """
    print("🚀 DÉBUT DE LA COMPARAISON MULTI-MÉTHODES")
    print("="*70)

    methods = ['gasf', 'mtf', 'rp']
    all_results = {}

    # Entraîner chaque méthode
    for method in methods:
        try:
            results = train_and_evaluate_method(method)
            if results:
                all_results[method] = results
            else:
                print(f"⚠️ Échec pour {method}")
        except Exception as e:
            print(f"❌ Erreur pour {method}: {e}")

    # Générer le rapport de comparaison
    if all_results:
        generate_comparison_report(all_results)
    else:
        print("❌ Aucun résultat à comparer")

    return all_results

In [ ]:
def generate_comparison_report(all_results):
    """
    Rapport de comparaison AMÉLIORÉ
    """
    print("\n" + "="*80)
    print("📊 RAPPORT DE COMPARAISON ResNet-CBAM FINAL")
    print("="*80)

    print("\n🏆 PERFORMANCES PAR MÉTHODE (ResNet-CBAM):")
    print("-" * 80)
    print(f"{'Méthode':<8} {'Accuracy':<10} {'F1-Macro':<10} {'F1-Crise':<10} {'Precision-C':<12} {'Recall-C':<10}")
    print("-" * 80)

    best_accuracy = 0
    best_method = ""

    for method, results in all_results.items():
        print(f"{method.upper():<8} {results['accuracy']:<10.4f} {results['f1_macro']:<10.4f} "
              f"{results['f1_crise']:<10.4f} {results['precision_crise']:<12.4f} "
              f"{results['recall_crise']:<10.4f}")

        if results['accuracy'] > best_accuracy:
            best_accuracy = results['accuracy']
            best_method = method

    print("-" * 80)
    print(f"🥇 MEILLEURE MÉTHODE: {best_method.upper()} avec {best_accuracy:.4f} accuracy")

    # Comparaison avec objectif
    print(f"\n📈 ANALYSE DES AMÉLIORATIONS:")
    print("-" * 40)
    print(f"Objectif visé: 97.0% accuracy")

    for method, results in all_results.items():
        improvement = (results['accuracy'] - 0.93) * 100  # Amélioration vs baseline 93%
        target_gap = (0.97 - results['accuracy']) * 100   # Écart vs objectif 97%

        print(f"{method.upper()}:")
        print(f"  Amélioration vs baseline: +{improvement:.1f} points")
        if results['accuracy'] >= 0.97:
            print(f"  🎯 OBJECTIF ATTEINT!")
        else:
            print(f"  Écart vs objectif: -{target_gap:.1f} points")

    # Visualisation (votre code existant)
    create_comparison_plots(all_results)

In [ ]:
def create_comparison_plots(all_results):
    """
    Crée des graphiques de comparaison
    """
    if len(all_results) < 2:
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    methods = list(all_results.keys())
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    # 1. Comparaison des métriques principales
    metrics = ['accuracy', 'f1_macro', 'precision_crise', 'recall_crise']
    metric_names = ['Accuracy', 'F1-Macro', 'Precision Crise', 'Recall Crise']

    for i, (metric, name) in enumerate(zip(metrics, metric_names)):
        values = [all_results[method][metric] for method in methods]
        bars = axes[i//2, i%2].bar(methods, values, color=colors[:len(methods)])
        axes[i//2, i%2].set_title(f'Comparaison {name}')
        axes[i//2, i%2].set_ylabel(name)
        axes[i//2, i%2].set_ylim(0, 1)

        # Ajouter les valeurs sur les barres
        for bar, value in zip(bars, values):
            axes[i//2, i%2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                                f'{value:.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(PLOTS_PATH + 'multi_method_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📈 Graphiques sauvegardés dans: {PLOTS_PATH}multi_method_comparison.png")

In [ ]:
# Script principal
if __name__ == "__main__":
    # Créer les dossiers nécessaires
    os.makedirs(MODELS_PATH, exist_ok=True)
    os.makedirs(PLOTS_PATH, exist_ok=True)

    # Lancer la comparaison multi-méthodes
    results = compare_all_methods()

    print("\n🎉 ANALYSE MULTI-MÉTHODES TERMINÉE!")
    print("="*50)
    print("📁 Fichiers générés:")
    print("   📈 Graphiques de comparaison")
    print("   🤖 Modèles entraînés pour chaque méthode")
    print("   📊 Rapports de performance détaillés")